In [1]:
import os
import numpy as np
from tokenizers import Tokenizer
TOKENIZER_FILE = "784million.json"
INPUT_TEXT_FILE = "corpus.txt"
OUTPUT_DIR = "."

def prepare():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if not os.path.exists(TOKENIZER_FILE):
        if os.path.exists(os.path.join("..", TOKENIZER_FILE)):
            tok_path = os.path.join("..", TOKENIZER_FILE)
        else:
            raise FileNotFoundError(f"Could not find '{TOKENIZER_FILE}'. Please put your tokenizer JSON file in this directory.")
    else:
        tok_path = TOKENIZER_FILE

    print(f"Loading BERT tokenizer from '{tok_path}'...")
    tokenizer = Tokenizer.from_file(tok_path)
    vocab_size = tokenizer.get_vocab_size()
    print(f"Tokenizer loaded successfully! Vocab size: {vocab_size:,}")

    # Check input file
    input_path = INPUT_TEXT_FILE
    if not os.path.exists(input_path):
        if os.path.exists(os.path.join("..", "Vectora-v1.1", "train.txt")):
            input_path = os.path.join("..", "Vectora-v1.1", "train.txt")
        elif os.path.exists(os.path.join("..", "..", "vectora_dataset", "train.txt")):
            input_path = os.path.join("..", "..", "vectora_dataset", "train.txt")
        else:
            raise FileNotFoundError(f"Could not find '{INPUT_TEXT_FILE}'.")

    print(f"Reading and tokenizing text from: {input_path}")
    train_bin = os.path.join(OUTPUT_DIR, "train.bin")
    val_bin = os.path.join(OUTPUT_DIR, "val.bin")

    total_tokens = 0
    buffer = []
    chunk_size = 500_000 
    with open(train_bin, "wb") as f_train, open(val_bin, "wb") as f_val:
        with open(input_path, "r", encoding="utf-8", errors="ignore") as f:
            for line_idx, line in enumerate(f):
                if not line.strip():
                    continue
                ids = tokenizer.encode(line).ids
                buffer.extend(ids)
                if len(buffer) >= chunk_size:
                    arr = np.array(buffer, dtype=np.uint16)
                    # 90% train, 10% val
                    split_idx = int(len(arr) * 0.90)
                    arr[:split_idx].tofile(f_train)
                    arr[split_idx:].tofile(f_val)
                    total_tokens += len(arr)
                    print(f"Tokenized {total_tokens:,} tokens so far...", end="\r")
                    buffer = []
        if buffer:
            arr = np.array(buffer, dtype=np.uint16)
            split_idx = int(len(arr) * 0.90)
            arr[:split_idx].tofile(f_train)
            arr[split_idx:].tofile(f_val)
            total_tokens += len(arr)
    print(f"\nDone! Successfully tokenized {total_tokens:,} tokens.")
    print(f"Saved: {train_bin} ({os.path.getsize(train_bin) / (1024*1024):.2f} MB)")
    print(f"Saved: {val_bin} ({os.path.getsize(val_bin) / (1024*1024):.2f} MB)")
if __name__ == "__main__":
    prepare()


Loading BERT tokenizer from '784million.json'...
Tokenizer loaded successfully! Vocab size: 50,000
Reading and tokenizing text from: corpus.txt
Tokenized 189,545,927 tokens so far...
Done! Successfully tokenized 189,563,561 tokens.
Saved: .\train.bin (325.41 MB)
Saved: .\val.bin (36.16 MB)
